In [ ]:
%%writefile ai_dashboard.py
import streamlit as st
import pandas as pd
from groq import Groq


if 'archetype_target' not in st.session_state:
    st.session_state.archetype_target = "All Properties"
if 'map_target' not in st.session_state:
    st.session_state.map_target = "📍 Show all buildings in this cluster"

if 'last_response' not in st.session_state:
    st.session_state.last_response = ""
if 'mentioned_props' not in st.session_state:
    st.session_state.mentioned_props = []

def jump_to_location(map_label):
    st.session_state.archetype_target = "All Properties"
    st.session_state.map_target = map_label


df = pd.read_csv("leaseup_dashboard_data.csv")

market_txt = df.groupby('Market')['LeaseUpTime'].mean().round(2).to_string()
archetype_txt = df.groupby('Archetype')[['LeaseUpTime', 'Premium_to_Submarket_Pct']].mean().round(3).to_string()
submarket_txt = df.groupby('Submarket')['Premium_to_Submarket_Pct'].mean().sort_values(ascending=False).head(5).round(3).to_string()

top_props = df.sort_values('Premium_to_Submarket_Pct', ascending=False).groupby('Archetype').head(5)
top_props_txt = top_props[['Archetype', 'Submarket', 'Name']].to_string(index=False)

data_summary = f"""
Here is the Real Estate Investment Data Summary:

Overall Portfolio:
- Total Properties Analyzed: {len(df)}
- Average Lease-Up Time: {df['LeaseUpTime'].mean():.1f} months

Performance by Archetype:
{archetype_txt}

Top 5 Highest Yielding Submarkets (Districts):
{submarket_txt}

Top Recommended Properties by Archetype and Submarket:
{top_props_txt}
"""


st.set_page_config(page_title="Investment AI Assistant", layout="wide")
st.title("🏙️ Real Estate Investment Dashboard")

with st.sidebar:
    st.header("Market Stats")
    st.dataframe(df.groupby('Archetype')['LeaseUpTime'].mean())
    st.write("---")
    st.write("Data processed and ready for NLP analysis.")
    
    # FALLBACK LOGIC: Works on Cloud AND Local Mac!
    try:
        api_key = st.secrets["GROQ_API_KEY"]
    except:
        st.warning("Local Mode: Cloud Vault not found.")
        api_key = st.text_input("Paste Groq API Key to test locally:", type="password")

st.subheader("📍 Geotag Board: Locality Explorer")
st.write("Filter by investment strategy and jump directly to specific properties on the map.")

filter_col, jump_col, map_col = st.columns([1, 1, 3])
df_coords = df.dropna(subset=['Latitude', 'Longitude', 'Name']).copy()
df_coords['MapLabel'] = df_coords['Name'] + " (" + df_coords['Submarket'] + ")"

with filter_col:
    options = ["All Properties"] + list(df['Archetype'].dropna().unique())
    st.radio("Step 1: Filter by Strategy:", options, key="archetype_target")

if st.session_state.archetype_target == "All Properties":
    type_filtered_df = df_coords.copy()
else:
    type_filtered_df = df_coords[df_coords['Archetype'] == st.session_state.archetype_target].copy()

with jump_col:
    jump_options = ["📍 Show all buildings in this cluster"] + sorted(type_filtered_df['MapLabel'].tolist())
    
    if st.session_state.map_target not in jump_options:
        st.session_state.map_target = "📍 Show all buildings in this cluster"
        
    st.selectbox("Step 2: Select a building:", jump_options, key="map_target")

with map_col:
    if st.session_state.map_target == "📍 Show all buildings in this cluster":
        map_df = type_filtered_df.copy()
    else:
        map_df = type_filtered_df[type_filtered_df['MapLabel'] == st.session_state.map_target].copy()
    
    map_df = map_df.rename(columns={'Latitude': 'lat', 'Longitude': 'lon'})
    if not map_df.empty:
        map_df = map_df[['lat', 'lon']]
        st.map(map_df)
    else:
        st.warning("No GPS coordinates found for this selection.")


st.subheader("🤖 Ask the Investment Assistant")

with st.form(key="chat_form"):
    user_input = st.text_input("Example: What is the best submarket to rent a Luxury High-Rise, and what are the top properties there?")
    submit_btn = st.form_submit_button("Ask Assistant")

if submit_btn and user_input:
    if not api_key:
        st.error("Please provide an API key in the sidebar to use the AI.")
    else:
        with st.spinner("Analyzing market data..."):
            prompt = f"""
            You are an expert Real Estate Investment Analyst. Use the following data summary to answer the user's question.
            
            DATA CONTEXT:
            {data_summary}
            
            USER QUESTION:
            {user_input}
            
            RESPONSE GUIDELINE:
            - Be concise, professional, and highly specific.
            - If asked for recommendations, rank the top Submarkets and list the SPECIFIC property names.
            - Do not hallucinate.
            """
            
            try:
                # Connected to the updated model
                client = Groq(api_key=api_key)
                response = client.chat.completions.create(
                    model="llama-3.1-8b-instant",
                    messages=[{"role": "user", "content": prompt}]
                )
                
                st.session_state.last_response = response.choices[0].message.content
                
                # Scan for properties
                mentioned = []
                for label in df_coords['MapLabel'].unique():
                    name_only = label.split(" (")[0] 
                    if name_only in st.session_state.last_response:
                        mentioned.append(label)
                st.session_state.mentioned_props = mentioned
                
            except Exception as e:
                st.error(f"API Error: {e}")

if st.session_state.last_response:
    st.markdown("### Assistant Response:")
    st.write(st.session_state.last_response)
    
    if st.session_state.mentioned_props:
        st.write("---")
        st.write("**📍 Jump to mentioned properties on the map:**")
        
        btn_cols = st.columns(min(len(st.session_state.mentioned_props), 4)) 
        for i, label in enumerate(st.session_state.mentioned_props):
            name_only = label.split(" (")[0]
            with btn_cols[i % len(btn_cols)]:
                st.button(f"🗺️ View {name_only}", on_click=jump_to_location, args=(label,), key=f"btn_{i}")

Overwriting ai_dashboard.py


In [ ]:
!streamlit run ai_dashboard.py


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://10.150.133.192:8501



Task 3. Include documentation on how you built the AI components and any prompt engineering techniques
used.

The interactive chat feature is powered by a cloud-hosted Large Language Model (LLM), enabling robust, public-facing deployment without reliance on local hardware. Specifically, the application utilizes Meta's Llama 3.1 (8B parameter) model, accessed via the ultra-fast Groq Cloud API (llama-3.1-8b-instant). The integration is handled through the groq Python library. To ensure security in a public cloud environment, the application utilizes Streamlit Secrets as an encrypted vault to manage API authentication. The system passes user queries and contextual dataframe summaries to the remote inference server and returns natural language insights.

Prompt Engineering Techniques Used:
To ensure the AI provides accurate, non-hallucinated, and highly specific real estate advice, several prompt engineering techniques were applied:

Context Injection (RAG-lite): Rather than relying on the LLM's base training data, the script aggregates live data using Pandas (calculating averages, grouping by archetypes, and sorting top properties) and injects it directly into the prompt via Python f-strings as a DATA CONTEXT block.

Structured Text Formatting: Raw Python dictionaries degrade LLM comprehension. We utilized Pandas' .to_string() method to convert complex dataframes into clean, Markdown-style tabular text. This dramatically improves the LLM's spatial awareness of the data, allowing it to accurately compare submarkets and rent premiums.

Role Prompting: The prompt initializes the AI with a specific persona: "You are an expert Real Estate Investment Analyst," ensuring the tone of the output matches the professional nature of the dashboard.

Task 5. Reflection: Advantages and Limitations of the AI-Enhanced Dashboard

The integration of a Large Language Model (LLM) into a traditional geospatial data dashboard presents a significant shift in how real estate analytics can be consumed. While this hybrid architecture offers powerful new capabilities, it also introduces specific technical constraints.

Advantages:

Intuitive Data Accessibility: Traditional dashboards require users to manually filter and interpret complex visual data. By integrating the Llama 3.1 model, the dashboard allows users to query the dataset using natural language. This drastically lowers the technical barrier to entry, allowing stakeholders to ask direct questions (e.g., "Where should I invest in a luxury high-rise?") and receive immediate, synthesized insights.

Decoupled Cloud Scalability: Migrating the LLM inference from a local environment (Ollama) to a cloud-based API (Groq) proved highly advantageous. It eliminated the heavy hardware requirements on the host machine, allowing the application to be deployed globally via Streamlit Community Cloud with near-instant inference speeds.

Cohesive User Experience (UX): The implementation of session state memory to dynamically generate interactive map buttons based on the AI’s text output bridges the gap between the NLP interface and the visual map, creating a seamless, actionable tool rather than a static chat window.

Limitations:

Context Window Constraints: LLMs cannot ingest massive, raw CSV files efficiently due to token limits and degrading recall. To bypass this, the architecture relies on a "RAG-lite" approach by pre-calculating averages and top properties using Pandas before injecting them into the prompt. Consequently, the AI can only analyze the summarized data it is fed, rather than the entire raw dataset.

API Dependency: Relying on a cloud API introduces an external point of failure. If the Groq API experiences downtime, rate limiting, or model deprecations (as seen during development), the core NLP functionality of the application breaks.

Hallucination Risks: Despite rigorous prompt engineering and strict guardrails instructing the model to rely only on the provided context, the inherent nature of generative AI means the risk of data hallucination is never truly zero. Outputs must still be cross-referenced with the raw data.

Gen AI use Documentation

What the Code is Doing?

The code in Task_2.ipynb bridges the gap between statistical analysis and the final interactive product. First, it applies K-Means clustering (an unsupervised machine learning algorithm) to segment the properties based on their yield and lease-up metrics. The code groups these properties into distinct, strategic archetypes (e.g., Suburban High-Velocity, Urban Luxury High-Rise). After assigning these categories, the code exports the finalized dataset as leaseup_dashboard_data.csv. Finally, the notebook utilizes Jupyter magic commands (%%writefile) to write and compile the foundational Python script (ai_dashboard.py) required to launch the Streamlit web application.

How and Why Generative AI Was Used?

For this task, Generative AI served as both an analytical sounding board and a deployment architect. Analytically, I used the AI to help interpret the mathematical outputs of the K-Means clusters, which assisted in formulating accurate, business-relevant names for the property archetypes. From an engineering perspective, AI was crucial for navigating the complexities of building a web application. I used the AI to troubleshoot Streamlit's specific rendering lifecycle, acquiring the code necessary to implement "session state" memory so the interactive map would not reset the application. Furthermore, the AI guided the critical architectural pivot from a local Ollama LLM to the Groq Cloud API, providing the updated code blocks required to deploy the dashboard successfully to the public internet.